In [13]:
import pandas as pd

In [ ]:
def preprocess_peptide_dataset(csv_path, min_samples_per_cell_line=10):
    """
    Предобработка датасета пептидов для soft-prompt tuning
    """
    # 1. Загрузка датасета
    print(f"Загрузка датасета из {csv_path}...")
    df = pd.read_csv(csv_path)
    print(f"Исходный размер датасета: {len(df)} строк")
    
    # 2. Оставляем только нужные колонки
    df = df[['sequence', 'is_cpp', 'cell_line']].copy()
    
    # 3. Приведение последовательностей к единому виду
    print("\nОбработка последовательностей...")
    df['sequence'] = df['sequence'].astype(str).str.upper()
    
    # Определяем разрешенные символы
    allowed_chars = set("ACDEFGHIKLMNPQRSTVWYXZBJ")
    
    def is_valid_sequence(seq, allowed_set):
        """Проверяет, состоит ли последовательность только из разрешенных символов."""
        if not isinstance(seq, str) or not seq:
            return False
        return all(char in allowed_set for char in seq)
    
    # Фильтрация последовательностей
    df['is_valid'] = df['sequence'].apply(lambda x: is_valid_sequence(x, allowed_chars))
    df_filtered = df[df['is_valid']].copy()
    df_filtered = df_filtered.drop(columns=['is_valid'])
    
    print(f"После фильтрации по допустимым символам: {len(df_filtered)} строк")
    print(f"Отфильтровано: {len(df) - len(df_filtered)} строк")
    
    # 4. Обработка клеточных линий
    print(f"\nОбработка клеточных линий (минимум образцов: {min_samples_per_cell_line})...")
    
    # Подсчет образцов для каждой клеточной линии
    cell_line_counts = df_filtered['cell_line'].value_counts()
    print("\nРаспределение по клеточным линиям до объединения:")
    for cell_line, count in cell_line_counts.items():
        print(f"  {cell_line}: {count} образцов")
    
    # Определяем редкие клеточные линии
    rare_cell_lines = cell_line_counts[cell_line_counts < min_samples_per_cell_line].index.tolist()
    
    if rare_cell_lines:
        print(f"\nКлеточные линии с < {min_samples_per_cell_line} образцов (будут объединены в 'other'):")
        for cl in rare_cell_lines:
            print(f"  {cl}: {cell_line_counts[cl]} образцов")
        
        # Заменяем редкие клеточные линии на "other"
        df_filtered.loc[df_filtered['cell_line'].isin(rare_cell_lines), 'cell_line'] = 'other'
    
    # Финальное распределение
    final_cell_line_counts = df_filtered['cell_line'].value_counts()
    print("\nФинальное распределение по клеточным линиям:")
    for cell_line, count in final_cell_line_counts.items():
        print(f"  {cell_line}: {count} образцов")
    
    # 5. Статистика по CPP
    print("\nСтатистика по CPP:")
    cpp_counts = df_filtered['is_cpp'].value_counts()
    print(f"  CPP пептиды (is_cpp=1): {cpp_counts.get(1, 0)} образцов")
    print(f"  Не-CPP пептиды (is_cpp=0): {cpp_counts.get(0, 0)} образцов")
    
    # Дополнительная статистика для CPP по клеточным линиям
    cpp_df = df_filtered[df_filtered['is_cpp'] == 1]
    if len(cpp_df) > 0:
        print("\nРаспределение CPP пептидов по клеточным линиям:")
        cpp_cell_lines = cpp_df['cell_line'].value_counts()
        for cell_line, count in cpp_cell_lines.items():
            print(f"  {cell_line}: {count} CPP пептидов")
    
    return df_filtered

def perform_oversampling(df, target_samples):
    """Выполняет оверсэмплинг для каждого класса до target_samples."""
    
    # Группируем по клеточным линиям
    grouped = df.groupby('cell_line')
    
    resampled_dfs = []
    for name, group in grouped:
        if len(group) < target_samples:
            # Дублируем случайные строки, чтобы достичь нужного количества
            resampled_group = group.sample(n=target_samples, replace=True, random_state=42)
            resampled_dfs.append(resampled_group)
        else:
            # Если класс уже достаточно большой, оставляем его как есть
            resampled_dfs.append(group)
            
    # Собираем все обратно в один датафрейм и перемешиваем
    df_resampled = pd.concat(resampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_resampled

def format_sequences_for_protgpt2(df, seq_column='sequence'):
    """
    Преобразует последовательности в формат ProtGPT2:
    - Добавляет <|endoftext|> в начало
    - Добавляет переводы строк каждые 60 аминокислот
    
    Args:
        df: DataFrame с последовательностями
        seq_column: название колонки с последовательностями
    
    Returns:
        DataFrame с преобразованными последовательностями
    """
    df = df.copy()
    
    def format_sequence(seq):
        # Добавляем токен в начало
        formatted = "<|endoftext|>"
        
        # Разбиваем на строки по 60 символов
        for i in range(0, len(seq), 60):
            formatted += "\n" + seq[i:i+60]
        
        # НЕ добавляем <|endoftext|> в конец для обучения
        # (он нужен только для расчета perplexity)
        
        return formatted
    
    # Применяем форматирование
    df[seq_column] = df[seq_column].apply(format_sequence)
    
    return df

In [11]:
path = 'C:/Users/ALI/itmo-cpp/input_data/all_peptides_for_classification.csv'

In [12]:
processed_df = preprocess_peptide_dataset(path, min_samples_per_cell_line=10)

Загрузка датасета из C:/Users/ALI/itmo-cpp/input_data/all_peptides_for_classification.csv...
Исходный размер датасета: 2922 строк

Обработка последовательностей...
После фильтрации по допустимым символам: 2624 строк
Отфильтровано: 298 строк

Обработка клеточных линий (минимум образцов: 10)...

Распределение по клеточным линиям до объединения:
  HeLa cells: 197 образцов
  NIH-3T3 cells: 62 образцов
  A549 cells: 44 образцов
  CHO cells: 39 образцов
  CHO-K1 cells: 36 образцов
  MCF7 cells: 34 образцов
  HaCaT cells: 34 образцов
  MDA-MB-231 cells: 32 образцов
  Jurkat cells: 24 образцов
  Aortic endothelial cells: 23 образцов
  Human bowes melanoma cells: 21 образцов
  U87 cells: 20 образцов
  HEK293 cells: 19 образцов
  RAW264.7 cells: 11 образцов
  U373 MG cells: 10 образцов
  A375 cells: 10 образцов
  HEK293T cells: 9 образцов
  DAMI cells: 9 образцов
  eGFP HeLa 654: 7 образцов
  DU-145: 7 образцов
  ARPE-19 cells: 6 образцов
  HepG2 cells: 6 образцов
  Saos-2 cells: 6 образцов
  N.

C:\Users\ALI\AppData\Local\Temp\ipykernel_20048\4095064632.py:65: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"  CPP пептиды (is_cpp=1): {cpp_counts.get(1, 0)} образцов")
C:\Users\ALI\AppData\Local\Temp\ipykernel_20048\4095064632.py:66: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"  Не-CPP пептиды (is_cpp=0): {cpp_counts.get(0, 0)} образцов")


In [ ]:
cpp_df = processed_df[processed_df['is_cpp'] == 1].copy()

# Определяем целевое количество сэмплов
target_count = cpp_df['cell_line'].value_counts().max()
print(f"\nВыполняется оверсэмплинг минорных классов до {target_count} образцов...")

# Применяем оверсэмплинг
df_train_balanced = perform_oversampling(cpp_df, target_count)

print("\nРазмер датасета после оверсэмплинга:")
print(df_train_balanced['cell_line'].value_counts())
print(f"Итоговый размер обучающего датасета: {len(df_train_balanced)} строк")


Выполняется оверсэмплинг минорных классов до 167 образцов...

Размер датасета после оверсэмплинга:
cell_line
Aortic endothelial cells      167
HaCaT cells                   167
A375 cells                    167
MDA-MB-231 cells              167
RAW264.7 cells                167
CHO-K1 cells                  167
A549 cells                    167
MCF7 cells                    167
other                         167
HeLa cells                    167
CHO cells                     167
NIH-3T3 cells                 167
HEK293 cells                  167
Jurkat cells                  167
Human bowes melanoma cells    167
U373 MG cells                 167
U87 cells                     167
Name: count, dtype: int64
Итоговый размер обучающего датасета: 2839 строк


In [19]:
# Форматируем для ProtGPT2
df_train_balanced = format_sequences_for_protgpt2(df_train_balanced)

In [20]:
df_train_balanced.to_csv('cpp_only_167.csv', index=False)